In [1]:
import torch
from torch import nn
from torchrl.envs import PettingZooWrapper
from pettingzooenv import MahjongGameEnv
from torchrl.envs import TransformedEnv
from torchrl.envs.transforms import ActionMask
from torchrl.envs.utils import MarlGroupMapType
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
# from pygame_visualizer import render_game_state
# import pygame
# import time
# pygame.init()
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)

env = PettingZooWrapper(env=MahjongGameEnv(), use_mask=True, return_state=True, categorical_actions=True, group_map=MarlGroupMapType.ALL_IN_ONE_GROUP)

# screen.fill('white')
# render_game_state(env._env.gamestate, screen, font)
# pygame.display.update()
# time.sleep(100)

print(env.full_observation_spec_unbatched)

2026-07-20 18:12:32,842	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


Composite(
    agents: Composite(
        observation: Composite(
            observation: UnboundedDiscrete(
                shape=torch.Size([4, 156, 46]),
                space=ContinuousBox(
                    low=Tensor(shape=torch.Size([4, 156, 46]), device=cpu, dtype=torch.uint8, contiguous=True),
                    high=Tensor(shape=torch.Size([4, 156, 46]), device=cpu, dtype=torch.uint8, contiguous=True)),
                device=cpu,
                dtype=torch.uint8,
                domain=discrete),
            device=None,
            shape=torch.Size([4]),
            data_cls=None),
        action_mask: Categorical(
            shape=torch.Size([4, 75]),
            space=CategoricalBox(n=2),
            device=cpu,
            dtype=torch.bool,
            domain=discrete),
        mask: Categorical(
            shape=torch.Size([4]),
            space=CategoricalBox(n=2),
            device=cpu,
            dtype=torch.bool,
            domain=discrete),
        devic

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()   # or .to(torch.float32)

policy_net = nn.Sequential(
    nn.Flatten(1),
    CastToFloat(),
    MultiAgentMLP(
        n_agent_inputs = 156 * 46,       
        n_agent_outputs = 75,       
        n_agents = 4,
        centralized = False,        
        share_params = True,      
        depth = 10,               
        num_cells = 1024     
    )
)

policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)  # we'll need the log-prob for the PPO loss


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tensordict\_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_out = _set_tensor_dict(


In [4]:
tensordict_data = env.rollout(max_steps=100, policy=policy)

In [5]:
from pygame_visualizer import render_game_state
import pygame
import time
pygame.init()
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
from sys import exit
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()
    screen.fill('white')
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()

SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


{'player_0': 0, 'player_1': 0, 'player_2': 0, 'player_3': 0}